In [ ]:
# Fetch

import os
import sys 
from pathlib import Path 

wd = Path(os.getcwd())
root = wd.parent
sys.path.append(f"{root}/modules")

from fbref_module import *

import pandas as pd

countries = ["ENG", "ESP", "GER", "ITA", "AUT", "HUN"]
seasons = ["2022-2023", "2023-2024", "2024-2025"]

gl_list = []
for cc in countries:
    for s in seasons:
        gl_season = get_gamelog(cc, s)
        gl_season['Country'] = cc
        gl_season['Season'] = s
        gl_list.append(gl_season)

gamelogs_raw = pd.concat(gl_list, axis=0)
gamelogs_raw.shape

In [ ]:
# Clean gamelogs

import numpy as np

gamelogs = gamelogs_raw[["Country", "Season", "Wk", "Date", "Home", "Away", "Score"]].dropna(subset=["Score"]).copy()
gamelogs = gamelogs[gamelogs["Wk"] != "Wk"]
gamelogs[['Score_Home', 'Score_Away']] = gamelogs['Score'].str.split('–', expand=True)
gamelogs.drop(columns="Score", inplace=True)
gamelogs["Points_Home"] = np.where(gamelogs.Score_Home > gamelogs.Score_Away, 3, 
                                   np.where(gamelogs.Score_Home == gamelogs.Score_Away, 1, 0)
                                   )
gamelogs["Points_Away"] = np.where(gamelogs.Score_Home > gamelogs.Score_Away, 0, 
                                   np.where(gamelogs.Score_Home == gamelogs.Score_Away, 1, 3)
                                   )

gamelogs[["Wk", "Score_Home", "Score_Away", "Points_Home", "Points_Away"]] = gamelogs[["Wk", "Score_Home", "Score_Away", "Points_Home", "Points_Away"]].astype(int)

gamelogs

In [ ]:
# Create tables

import pandas as pd

def build_standings(gamelogs: pd.DataFrame):
    """
    Visszatér egy 3-szintű dict-tel:
    country → season → week → standings_df
    """

    # --- TÍPUSKONVERZIÓ: kötelező! ---
    numeric_cols = ["Score_Home", "Score_Away", "Points_Home", "Points_Away"]
    gamelogs[numeric_cols] = gamelogs[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    result = {}

    # Rendezés logikai sorrendben
    gamelogs = gamelogs.sort_values(["Country", "Season", "Wk", "Date"]).copy()

    for country, df_c in gamelogs.groupby("Country"):
        result[country] = {}
        for season, df_s in df_c.groupby("Season"):
            result[country][season] = {}

            for week in sorted(df_s["Wk"].unique()):
                df_until_week = df_s[df_s["Wk"] <= week]

                # Home statok
                home = df_until_week.groupby("Home").agg(
                    Points=("Points_Home", "sum"),
                    GF=("Score_Home", "sum"),
                    GA=("Score_Away", "sum"),
                )

                # Away statok
                away = df_until_week.groupby("Away").agg(
                    Points=("Points_Away", "sum"),
                    GF=("Score_Away", "sum"),
                    GA=("Score_Home", "sum"),
                )

                # Combine (fontos: fill_value=0 biztosítja hogy int maradjon)
                standings = (
                    home.rename_axis("Team")
                    .add(away.rename_axis("Team"), fill_value=0)
                    .reset_index()
                )

                standings["GD"] = standings["GF"] - standings["GA"]

                # Tabella rendezés
                standings = standings.sort_values(
                    ["Points", "GD", "GF"],
                    ascending=[False, False, False]
                ).reset_index(drop=True)

                result[country][season][week] = standings

    return result


tables = build_standings(gamelogs)


In [ ]:
tables["HUN"]["2024-2025"][32]

In [ ]:
# Upset loss-ok

def compute_upset_ratio(gamelogs, tables):
    """
    Visszaadja:
    {
        country : {
            season : DataFrame(Team, Lost_points, Possible_points, Ratio)
        }
    }
    """

    result = {}

    # biztos sorrend
    gamelogs = gamelogs.sort_values(["Country", "Season", "Wk", "Date"])

    for country, df_c in gamelogs.groupby("Country"):
        result[country] = {}

        for season, df_s in df_c.groupby("Season"):

            teams = pd.concat([df_s["Home"], df_s["Away"]]).unique()

            lost_points = {team: 0 for team in teams}
            possible_points = {team: 0 for team in teams}

            for _, row in df_s.iterrows():
                week = row["Wk"]
                if week < 3:
                    continue

                home, away = row["Home"], row["Away"]
                sh, sa = row["Score_Home"], row["Score_Away"]

                # pontok a meccsen
                if sh > sa:
                    pts_home, pts_away = 3, 0
                elif sh < sa:
                    pts_home, pts_away = 0, 3
                else:
                    pts_home, pts_away = 1, 1

                prev_table = tables[country][season].get(week - 1)
                if prev_table is None:
                    continue

                prev_points = prev_table.set_index("Team")["Points"]

                # ha valaki nincs az előző heti tabellában → skip
                if home not in prev_points or away not in prev_points:
                    continue

                home_prev = prev_points[home]
                away_prev = prev_points[away]

                # Home vizsgálat
                if away_prev < home_prev:                # ellenfél gyengébb volt
                    possible_points[home] += 3
                    lost_points[home] += (3 - pts_home)

                # Away vizsgálat
                if home_prev < away_prev:
                    possible_points[away] += 3
                    lost_points[away] += (3 - pts_away)

            # DataFrame építése
            df_out = pd.DataFrame({
                "Team": teams,
                "Lost_points_vs_weaker": [lost_points[t] for t in teams],
                "Possible_points": [possible_points[t] for t in teams],
            })

            df_out["Ratio"] = df_out.apply(
                lambda r: r["Lost_points_vs_weaker"] / r["Possible_points"]
                if r["Possible_points"] > 0 else None,
                axis=1
            )

            # rendezés: ki bukja el a legtöbb kötelezőt
            df_out = df_out.sort_values("Ratio", ascending=False).reset_index(drop=True)

            result[country][season] = df_out

    return result


ratios = compute_upset_ratio(gamelogs, tables)

In [ ]:
for country in countries:
    for season in seasons:
        print(f"{country} {season}: {ratios[country][season]['Ratio'].std():.3f}")


In [ ]:
# Plot

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.font_manager as font_manager
from adjustText import adjust_text

def plot_upset_vs_points_adam_style(tables, ratios, outlier_threshold=0.1, my_font_path=None):
    """
    Scatter plot minden ország–szezon csapataira:
    X = upset-ratio, Y = normalizált végső pontarány (0–1)
    - Lineáris regresszió
    - Outlierekhez annotáció
    """
    
    # --- előkészítés ---
    df_plot = []

    for country in ratios:
        if (country == "AUT") or (country == "HUN"):
            continue
        for season in ratios[country]:
            df_ratio = ratios[country][season].set_index("Team")
            last_week = max(tables[country][season].keys())
            final_table = tables[country][season][last_week].set_index("Team")

            for team in df_ratio.index:
                if team not in final_table.index:
                    continue
                upset_ratio = df_ratio.loc[team, "Ratio"]
                final_points = final_table.loc[team, "Points"]
                final_points_ratio = final_points / (last_week * 3)
                if upset_ratio is None or np.isnan(upset_ratio):
                    continue
                df_plot.append({
                    "Country": country,
                    "Season": season,
                    "Team": team,
                    "UpsetRatio": upset_ratio,
                    "PointRatio": final_points_ratio
                })

    df_plot = pd.DataFrame(df_plot)
    display(df_plot[df_plot.Team == "Sevilla"])

    X = df_plot["UpsetRatio"].values.reshape(-1,1)
    y = df_plot["PointRatio"].values

    # --- regresszió ---
    reg = LinearRegression()
    reg.fit(X, y)
    y_pred = reg.predict(X)
    r2 = r2_score(y, y_pred)
    slope = reg.coef_[0]
    intercept = reg.intercept_

    print(f"Regression equation: y = {slope:.3f}x + {intercept:.3f}")
    print(f"R² = {r2:.3f}")

    # --- plot ---
    background_color = '#3c3d3d'
    fig, ax = plt.subplots(figsize=(10,7))
    fig.patch.set_facecolor(background_color)
    ax.set_facecolor(background_color)

    # színek országok szerint
    countries = df_plot["Country"].unique()
    palette = plt.get_cmap("tab10")
    colors = {c: palette(i % 10) for i, c in enumerate(countries)}

    for country in countries:
        df_c = df_plot[df_plot["Country"] == country]
        ax.scatter(df_c["UpsetRatio"], df_c["PointRatio"], 
                   color=colors[country], alpha=0.7, s=80, label=country)

    # regressziós vonal
    x_line = np.linspace(0, 1, 100)
    y_line = reg.predict(x_line.reshape(-1,1))
    ax.plot(x_line, y_line, color='orange', lw=2, label='Linear regression')

    # annotálás outlierekhez
    texts = []
    for _, row in df_plot.iterrows():
        y_diff = abs(row["PointRatio"] - (slope*row["UpsetRatio"] + intercept))
        if y_diff > outlier_threshold:
            t = ax.text(row["UpsetRatio"], row["PointRatio"], f"{row['Team']} {row['Season']}",
                        color='white', fontsize=9)
            texts.append(t)

    # nyilak és automatikus pozicionálás
    adjust_text(texts, arrowprops=dict(arrowstyle="->", color='white', lw=0.8), ax=ax)

    
    # tengelyek
    ax.set_xlabel("Upset-ratio", color='white')
    ax.set_ylabel("Season final points ratio", color='white')
    ax.set_title("Upset-ratio vs. Season Points Ratio", color='white', fontsize=16)

    ax.spines['bottom'].set_color('white')
    ax.spines['left'].set_color('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(colors='white')
    ax.grid(alpha=0.2)

    # legend
    ax.legend(facecolor=background_color, framealpha=0.5, edgecolor='white', labelcolor='white')

    # vízjel
    mycolor = '#5ECB43'
    if my_font_path is not None:
        my_font_props = font_manager.FontProperties(fname=my_font_path)
        fig.text(0.86, 0.95, 'ADAM JAKUS', color=mycolor, fontsize=18, fontproperties=my_font_props, ha='center')
    else:
        fig.text(0.86, 0.95, 'ADAM JAKUS', color=mycolor, fontsize=18, ha='center')

    plt.tight_layout()
    plt.show()


# Példa: saját fonttal (ha van)
my_font_path = f"{root}/helpers/Nexa-ExtraLight.ttf"
plot_upset_vs_points_adam_style(tables, ratios, outlier_threshold=0.15, my_font_path=my_font_path)
